### Project Ariadne Delta Logprob Datasets

To measure the correlation between the ID-deltas and the OOD-deltas, we've got to create both datasets
from the samples we've collected in the previous step. 

For the OOD-Data, we've collected ground truth samples from the Qwen2.5-7B basemodel and augmented them using Qwen3-8B. For the ID-Data, we've collected rollouts for each checkpoint $i \in \{0, 10, 20, ..., 80\}$ and distilled answers for which Qwen3-8B decided they contain a calculation error. This gives us pairs $(x_{\text{gt}}, x_{\text{aug}})$ for the OOD-data and pairs $(x_{\text{gt}}, x_{\text{error}})$ for the ID-samples.

The question is, whether $\rho = \text{corr}(\Delta_{\text{ID}}, \Delta_{\text{OOD}}) > 0$ 

In [1]:
import os
import numpy as np
import pandas as pd

In [2]:
with open('/u/rfechner/data/ariadne/id-outputs-verify.parquet', 'rb') as file:
    df_id = pd.read_parquet(file)

In [3]:
len(df_id)

12009

In [4]:
df_id.iloc[0]['responses'][0]

"<think>\nOkay, let's see. The question is asking whether the student's answer includes explicit verification of the intermediate or final result. The student converted (0,3) to polar coordinates and arrived at (3, π/2). \n\nFirst, I need to check if there's any part where the student verifies their answer. The student calculated r as 3, which is correct. Then for theta, since x is 0 and y is positive, they concluded theta is π/2. \n\nLooking through the answer, the student explains that tan inverse of (3/0) is undefined, which points to the positive y-axis. They mention that points on the positive y-axis have an angle of π/2. But does that count as verification? \n\nWait, the instructions say verification includes checking by plugging back into the equation or reiterating to check correctness. The student didn't actually plug the result back into the original equations to confirm. They just explained why theta is π/2 based on the position. \n\nSo, there's no explicit step where they v

In [5]:
# filter id dataset
def filter_in_distribution(df : pd.DataFrame) -> pd.DataFrame:
    """
        Parses and filters dataset.

        Responses of the in-distribution dataframe are of the shape:
        <think>...</think> ... #### {yes|no} ...
        We should filter out responses which contain a "#### yes"
    """
    def mapper(response : list[str]):
        r = response[0][-100:]
        try:
            i = r.rindex('####')
        except ValueError:
            return False
        return 'yes' in r[i:i+10]
        
    cond = df['responses'].apply(mapper)

    return df[cond]
df_id_filtered = filter_in_distribution(df_id)

In [7]:
from utils import load_rollouts
import pickle

RELOAD=False
if RELOAD:
    rollouts : list[pd.DataFrame] = load_rollouts()
    with open('rollouts.parquet', 'wb') as file:
        pickle.dump(obj=rollouts, file=file)
else:
    with open('rollouts.parquet', 'rb') as file:
        rollouts = pickle.load(file)

In [8]:
a = pd.concat(rollouts, axis=0)
a.columns

Index(['input', 'output', 'gts', 'score', 'step', 'reward', 'acc',
       'checkpoint'],
      dtype='object')

In [9]:
df_id_filtered.columns

Index(['prompt', 'step', 'old_index', 'responses'], dtype='object')

### ID frequency analysis (skipping Delta construction for now...)


In [13]:
contains_verification = df_id_filtered.groupby('step').apply(len, include_groups=False).to_numpy()
denominator = df_id.groupby('step').apply(len, include_groups=False).to_numpy()
share = contains_verification / denominator


In [ ]:
share

array([0.11975591, 0.08437271, 0.07027818, 0.09404849, 0.1183432 ,
       0.12125749, 0.13262999, 0.14692308, 0.14926527])

### OOD delta construction

For the out-of-distribution samples, we're going to match up the augmented samples with their un-augmented counterparts.
This is straight forward, as we can extract the correct response (ground truth) from the prompt of the augmented sample.

#### OOD Answers

Let's load in the generated answers, group them to the original ground truths and then create the ood-deltas dataset.

In [2]:
with open('/u/rfechner/data/ariadne/ood-outputs-per-checkpoint-verify.parquet', 'rb') as file:
    df_ood_gen = pd.read_parquet(file)

In [3]:
def filter_parsed_ood_samples(df : pd.DataFrame) -> pd.DataFrame:
    """
        We'll apply a simple heuristic to decide whether an answer is a good augmentation:
        0) Thinking must've finished.
        1) It must contain a '####' marker signaling the start of the answer.
        2) it must contain "\\boxed{" within the last 100 characters
    """
    def mapper(responses : list[str]) -> bool | str:
        r = responses[0] # only a single response per row
        if '</think>' not in r:
            return False
        try:
            i = r.rindex('####')
        except ValueError:
            return False
        if "boxed{" not in r[i:]:
            return False
        return r[i:].removeprefix('####').lstrip()

    df['parsed_response'] = df['responses'].apply(mapper)
    df = df[df['parsed_response'].astype(bool)]
    return df

df_ood_gen_parsed = filter_parsed_ood_samples(df_ood_gen.copy())

In [4]:
df_ood_gen_parsed.iloc[14]['prompt'][1]

{'content': "You're given a question and a correct student answer, your task is to inject verification behaviour (e.g., adding verification of arithmetic operations or a sanity check) if and only if the answer allows for a reasonable augmentation. IMPORTANT: Stay as close as possible to the correct answer. In case it is unreasonable to augment the correct answer with a verification, just return '#### Not applicable'. Otherwise return the complete verification-augmented answer, pre-pended by a '####'. Abstract example: If the Correct Answer is [reasoning] [solve step 1] ... [solve step k] ... [result], then your answer should be #### [reasoning] [solve step 1] ... [solve step k] ... [verification] [result], i.e. staying as close as possible to the initial answer, whilst introducing a final verification. \n\n\nQuestion:\nA group of $N$ students, where $N < 50$, is on a field trip. If their teacher puts them in groups of 8, the last group has 5 students. If their teacher instead puts them

In [5]:
def input_from_prompt(prompt : str) -> str:
    qprefix = "\nQuestion:\n"
    qindex = prompt.rindex(qprefix)
    aprefix = "\nSolution:\n"
    aindex = prompt.rindex(aprefix)
    question = prompt[qindex:aindex].removeprefix(qprefix)
    return question

def ground_truth_from_prompt(prompt : str) -> str:
    aprefix = "\nSolution:\n"
    aindex = prompt.rindex(aprefix)
    gt = prompt[aindex:].removeprefix(aprefix).removesuffix('\n')
    return gt

In [6]:
a = df_ood_gen_parsed.iloc[3]['prompt'][1]['content']
input_from_prompt(a), ground_truth_from_prompt(a)

('$n$ fair 6-sided dice are simultaneously rolled. The probability that exactly two of them show a number other than 1 is $\\frac{25}{216}$. Find $n$.',
 "To solve this problem, we need to calculate the probability that exactly two out of \\( n \\) dice show a number other than 1 when rolled simultaneously. Here's the step-by-step reasoning:\n\n1. **Probability of a Single Die Showing a Number Other Than 1**: Each die has 6 faces, and 5 of them are numbers other than 1. Therefore, the probability that a single die shows a number other than 1 is \\( \\frac{5}{6} \\).\n\n2. **Probability of a Single Die Showing a 1**: The probability that a single die shows a 1 is \\( \\frac{1}{6} \\).\n\n3. **Choosing Exactly Two Dice to Show a Number Other Than 1**: We need to choose 2 dice out of \\( n \\) to show a number other than 1. This can be done in \\( \\binom{n}{2} \\) ways.\n\n4. **Probability Calculation**: The probability that exactly two dice show a number other than 1 and the remaining \

In [7]:
df_ood_gen_parsed.iloc[3]['parsed_response']

"To solve this problem, we need to calculate the probability that exactly two out of $ n $ dice show a number other than 1 when rolled simultaneously. Here's the step-by-step reasoning:\n\n1. **Probability of a Single Die Showing a Number Other Than 1**: Each die has 6 faces, and 5 of them are numbers other than 1. Therefore, the probability that a single die shows a number other than 1 is $ \\frac{5}{6} $.\n\n2. **Probability of a Single Die Showing a 1**: The probability that a single die shows a 1 is $ \\frac{1}{6} $.\n\n3. **Choosing Exactly Two Dice to Show a Number Other Than 1**: We need to choose 2 dice out of $ n $ to show a number other than 1. This can be done in $ \\binom{n}{2} $ ways.\n\n4. **Probability Calculation**: The probability that exactly two dice show a number other than 1 and the remaining $ n-2 $ dice show a 1 is $ \\left(\\frac{5}{6}\\right)^2 \\left(\\frac{1}{6}\\right)^{n-2} $.\n\n5. **Combining Both**: The overall probability is given by multiplying the num

In [8]:
rows = []
for i, row in df_ood_gen_parsed.iterrows():
    prompt = row['prompt'][1]['content']
    inp, gt = input_from_prompt(prompt), ground_truth_from_prompt(prompt)
    step = row['step']
    verified_response = row['parsed_response']
    correct = {
        'prompt' : [{'role' : 'user', 'content' : inp}, 
                    {'role' : 'assistant', 'content' : gt}],
        'behaviour_type' : 'anchor',
        'step' : step,
        "uid" : i
    }
    incorrect = {
        'prompt' : [{'role' : 'user', 'content' : inp}, 
                    {'role' : 'assistant', 'content' : verified_response}],
        'behaviour_type' : 'verification',
        'step' : step,
        "uid" : i
    }
    rows.append(correct); rows.append(incorrect)

In [14]:
ood_dataframe = pd.DataFrame(rows)
ood_dataframe.head(1)

,prompt,behaviour_type,step,uid
0,"[{'role': 'user', 'content': '$\overline{BC}$ ...",anchor,0,4718


In [19]:
# at the time of writing this, I have to make the number of lines in the deltas dataframe I'm passing divisible by 8.
# It is what it is.
drop_last = len(ood_dataframe) % 8
ood_dataframe = ood_dataframe[-drop_last:]
assert len(ood_dataframe) % 8 == 0

In [20]:
len(ood_dataframe)

5288

In [21]:
path = "/u/rfechner/data/ariadne"
os.makedirs(path, exist_ok=True)

with open(os.path.join(path, 'ood-deltas-per-checkpoint-verify.jsonl'), 'w') as file:
    ood_dataframe.to_json(path_or_buf=file, lines=True, orient='records')